# Mapping Excel to SPOD
- Prerequisites: 
  - Anaconda packages: `pandas, openpyxl`

This script imports the Mapping Excel sheet and processes it into 
SPOD JSON format, which can be imported back.

## Excel Structure Requirements

Sheet with name 'Mapping' containing the [Table](https://support.microsoft.com/en-us/office/create-and-format-tables-e81aa349-b006-4f8a-9806-5af9df0ac664) 'mapping'.

The mapping table must containt the columns:

FQD -> IM:Entity + ':' IM:Attribute


## Result

## Structure

1. Define mapping between SPOD (json) and columns in the resulting Excel sheet

## Configuration
The following parameters has to be definded when running as regular python script

In [1]:
MODEL = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.regen.json'
MODEL = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json'

#MAPPING = 'testdata/Sika_Mapping_DE-2022-05-09.xlsx'
MAPPING = 'testdata/2022-05-20 Mapping France.xlsx'

## Check prerequisites

In [2]:
import sys
import logging
import os
import json
import shutil
import copy
import time
from pathlib import Path

In [3]:
# openpyxl
import openpyxl
from openpyxl.worksheet.table import Table
from openpyxl.utils import cell
from openpyxl.styles import PatternFill

In [4]:
from tqdm.autonotebook import tqdm

/var/folders/4w/2w5kn0_514d02xt_w5yzdp080000gn/T/ipykernel_94487/987820437.py:1: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [5]:
import jsonpath_ng as jsonpath

## Initialize logging

In [6]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/excel-mapping-{run_stamp}.log'

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")

file_handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger('ExcelMappingLogger')
logger.setLevel(logging.DEBUG)
logger.addHandler(file_handler)
logger.addHandler(console_log_handler)

business_logger = logging.getLogger('business')
business_logger.info('TestBusinessLog')

In [7]:
spod_file = Path(MODEL)
assert spod_file.is_file(), f"Cannot find SPOD file '{spod_file.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
logger.info(f"Loaded SPOD {spod['model']['name']} from '{spod_file.resolve()}' revision {spod['_imprint_'].get('git-revision','???')}")

INFO - Loaded SPOD Sika-IM from '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json' revision a7a7381


In [8]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    print(f"- {entry}: {len(spod[entry])}")

Loaded /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json
{'name': 'Sika-IM', 'type': 'logical', 'language': 'de', 'uc': 'stb', 'dc': '2021-10-26 12:05:29 UTC', 'um': 'SPOD', 'dm': '2022-05-18 18:03:35.940741'}
Version {'database': '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.db', 'created': '2022-05-27 01:38:29.682073', 'Modelversion': '1.9', 'hashvalue': 5801372704987061358, 'git-revision': 'a7a7381', 'comment': 'Entries ending with + represent denormalized data and are not checked for consistency while reading back'}
Languages: ['de', 'en', 'fr']
- entities: 91
- attributes: 141
- systems: 40
- columns: 943


## Use the fyayc SPOD library

### Tools path

In [9]:
LIBRARY = '../../pythonWork/pythonSource'
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

In [10]:
from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [11]:
translator = Translator('de')

# Changes in mapping.xlsx versus previous version
Compare new mapping.xlsx with previous version and update the SPOD accordingly.
Create a log of changes in JSON format, close to the SPOD structure.

In [12]:
source_excel_file = Path(MAPPING)
wb = openpyxl.load_workbook(source_excel_file)
assert wb['Mapping'] is not None, f"No sheet named 'Mapping' in workbook '{OLD_MAPPING}'"
logger.info(f"Loaded workbook {MAPPING}")

INFO - Loaded workbook testdata/2022-05-20 Mapping France.xlsx


## Create a duplicat to annotate import progress

In [13]:
import_result_excel = source_excel_file.with_stem(source_excel_file.stem + '_imported')
shutil.copy(source_excel_file, import_result_excel)
wb_report = openpyxl.load_workbook(import_result_excel)
logger.info(f"Writing import report to '{import_result_excel}'")

INFO - Writing import report to 'testdata/2022-05-20 Mapping France_imported.xlsx'


# Compare mapping sheet against SPOD 

In [14]:
mappings = wb['Mapping']
logger.info(f"Sheet {mappings} contains {mappings.tables.keys()}")
mapping_table = mappings.tables['mapping']

logger.info(f"Processing table '{mapping_table.displayName}' spanning {mapping_table.ref} with headers {mapping_table.headerRowCount}")
table_range_tuple = cell.range_boundaries(mapping_table.ref)

first_data_row = table_range_tuple[1] + mapping_table.headerRowCount
headers = mappings[table_range_tuple[1]]

INFO - Sheet <Worksheet "Mapping"> contains dict_keys(['mapping'])
INFO - Processing table 'mapping' spanning A1:U1317 with headers 1


In [15]:
mapping_report_sheet = wb_report['Mapping']

In [16]:
column_mapping = dict()

index = 0
for column in headers:
    name = column.value
    logger.info(f"Header '{name}'")
    column_mapping[name] = index
    index += 1

list(column_mapping.items())[:18], len(column_mapping)

INFO - Header 'Column1'
INFO - Header 'Column2'
INFO - Header 'Column3'
INFO - Header 'Bemerkungen Lea'
INFO - Header 'Questions'
INFO - Header 'ADDOK_MATCO / COFAQ'
INFO - Header 'BIGMAT_SIKA'
INFO - Header 'Chausson_Tarifs SIKA'
INFO - Header 'CMEM_SIKA'
INFO - Header 'Fichier'
INFO - Header 'GEDIMAT_Matrice_ARTICLE-SIKA'
INFO - Header 'GEDIMAT_TARIF'
INFO - Header 'LA PDB_Création'
INFO - Header 'POINTE P_SIKA_FRANCE'
INFO - Header 'PROLIANS_348-SIKA FRANCE'
INFO - Header 'CTX-LMFR-FR'
INFO - Header 'Champs Alkemics'
INFO - Header 'Champs SMARTREF'
INFO - Header 'BATI'
INFO - Header 'Bricoman'
INFO - Header 'MR BRICOLAGE'


([('Column1', 0),
  ('Column2', 1),
  ('Column3', 2),
  ('Bemerkungen Lea', 3),
  ('Questions', 4),
  ('ADDOK_MATCO / COFAQ', 5),
  ('BIGMAT_SIKA', 6),
  ('Chausson_Tarifs SIKA', 7),
  ('CMEM_SIKA', 8),
  ('Fichier', 9),
  ('GEDIMAT_Matrice_ARTICLE-SIKA', 10),
  ('GEDIMAT_TARIF', 11),
  ('LA PDB_Création', 12),
  ('POINTE P_SIKA_FRANCE', 13),
  ('PROLIANS_348-SIKA FRANCE', 14),
  ('CTX-LMFR-FR', 15),
  ('Champs Alkemics', 16),
  ('Champs SMARTREF', 17)],
 21)

In [17]:
column_mapping.get('Jedele')

## jspath templates to access attributes

In [18]:
columns = { 'Examples': '$.attributes["{attr}"].examples["en"]' }

for skey, system in spod['systems'].items():
    columns[system['name']] = '$.columns["{col}"]["name"]'

In [19]:
columns

{'Examples': '$.attributes["{attr}"].examples["en"]',
 'ADDOK_MATCO / COFAQ': '$.columns["{col}"]["name"]',
 'Amazon FR': '$.columns["{col}"]["name"]',
 'BATI': '$.columns["{col}"]["name"]',
 'Bigmat_SIKA': '$.columns["{col}"]["name"]',
 'Bricoman': '$.columns["{col}"]["name"]',
 'CMEM_SIKA': '$.columns["{col}"]["name"]',
 'CMEM_SIKA_200421': '$.columns["{col}"]["name"]',
 'COFAQ_sika_france-mp-brut_net': '$.columns["{col}"]["name"]',
 'CTX-LMFR-FR': '$.columns["{col}"]["name"]',
 'CXM Access': '$.columns["{col}"]["name"]',
 'CXM DPB Schnittstelle': '$.columns["{col}"]["name"]',
 'CXM ProductUp FR': '$.columns["{col}"]["name"]',
 'Champs Alkemics': '$.columns["{col}"]["name"]',
 'Champs SMARTREF': '$.columns["{col}"]["name"]',
 'Chausson_Tarifs SIKA': '$.columns["{col}"]["name"]',
 'Excel ADDOK_MATCO:01.02.2022 Tabelle: Correspondance Nom douaniere': '$.columns["{col}"]["name"]',
 'Excel ADDOK_MATCO:01.02.2022 Tabelle: Matrice Tabelle: Examples': '$.columns["{col}"]["name"]',
 'Excel A

In [20]:
def build_json_path(path: str, **kwargs) -> jsonpath.Child:
    expanded = path.format(**kwargs)
    path = jsonpath.parse(expanded)
    return path

In [21]:
path = build_json_path(columns['Examples'], attr='ATTR249')
path

Child(Child(Child(Child(Root(), Fields('attributes')), Fields('ATTR249')), Fields('examples')), Fields('en'))

In [22]:
r = path.find(spod)
if len(r) > 0:
    r[0].value

In [23]:
def get_aid_from_fqdn(value: str) -> str:
    if value is not None:
        if ':' in value:
            index = value.rfind(':')
            return value[index+1:]
        return value
    return None

In [24]:
def process_row(headers: tuple, row: tuple):
    
    attrid = get_aid_from_fqdn(row[column_mapping['AID']])
    
    for header in headers:
        index = header.col_idx - 1
        new_value = row[index]
        
        colmapping = columns.get(header.value)
        if colmapping is not None:
            jpath = build_json_path(colmapping, attr=attrid)
            hits = jpath.find(spod)
            
            # accept empty
            if len(hits) > 0 and new_value is not None:
                assert len(hits) == 1, f"Expecting only one precise hit, got: {hits}"
                current_value = hits[0].value
                if current_value != new_value:
                    logger.info(f"Updating current value '{current_value}' to '{new_value}'")
                    jpath.update(spod, new_value)
            else:
                logger.debug(f"Cell and new value are undefined")
        else:
            logger.debug(f"Column {header.value} = {new_value}")
        


In [25]:
sample_row = next(mappings.iter_rows(min_row=first_data_row, max_row=first_data_row + 1, values_only=True))
#process_row(headers, sample_row)

In [26]:
create_statement = f"Created by mapping import"
user = 'bue'
from datetime import datetime
import pytz

timestamp_now = datetime.now()
timestamp_now_utc = pytz.utc.localize(timestamp_now)

stamp = '2022-04-28 15:38:01 UTC'
change_stamp = '2022-04-28 15:38:01.1'

stamp = timestamp_now_utc.strftime('%Y-%m-%d %H:%M:%S %Z')
change_stamp = timestamp_now_utc.strftime('%Y-%m-%d %H:%M:%S.%f')

In [27]:
SOURCE_KEY = 'Excel-Mapping-Import-Notebook'

In [28]:
def neq(lhs, rhs) -> bool:
    if lhs is None and rhs is None:
        return False
    if lhs == rhs:
        return False
    if lhs is None and (isinstance(rhs, str) and len(rhs) < 1):
        return False
    if rhs is None and (isinstance(lhs, str) and len(lhs) < 1):
        return False
    return True

def update_column(column, name: str, tech: str, row: tuple, attributes: set) -> []:
    result = None
    message = []
    updated = copy.deepcopy(column)

    current_attributes = set(c['attributesmapped'])
    diff = current_attributes.symmetric_difference(attributes)
    if len(diff) > 0:
        updated['attributesmapped'] = list(attributes)
        result = updated
        message.append(f"Altered attribute mapping from {current_attributes} to {attributes}")
        
    if neq(name, column['name']) or neq(tech, column['interface_col_id']):        
        updated['name'] = name
        updated['um'] = user
        updated['dm'] = stamp
        result = updated
        message.append(f"Renamed from {column['name']} to {name}")
    return result, ' '.join(message)

In [29]:
def add_defaults(element: dict):
    values = { 'um': None, 'dm': None, 'minzoomlevel': None, 'maxzoomlevel': None, 'publstatus': None,
             'referencedby': [], 'userdefprops': {} }
    element.update(values)
    return element

In [30]:
from SSOT_db.IM_JSON.jsattribute import attr2js

In [31]:
new_table_cache = dict()
new_column_cache = dict()
new_attribute_cache = dict()

def negative_number_generator() -> int:
    counter = 0
    while True:
        counter -= 1
        yield counter
    
new_element_id_generator = negative_number_generator()

default_domain = next(filter(lambda d: next(iter(d[1]['name'].values())) == 'Unknown', spod['domains'].items()))[0]

assert default_domain is not None
print(f"Default domain is {default_domain}")

ual = list(filter(lambda t: t[1]['shortname'] == 'unassigned', spod['entities'].items()))
assert len(ual) == 1, "Expecting exactly one entity named 'unassigned'"
default_entity = ual[0][0]

def process_system_row(spod: dict, system_key: str, key, name, tech, row):
    column_key = None
    column = None
    
    result = []
    
    # Mapped columns on this line
    aonline = list(filter(lambda cell: cell.value is not None, row[5:]))
    nonempty_cells_values = list(map(lambda c: c.value, aonline))
 
    if key is not None:
        index = key.rfind(':')
        colkey = key[index+1:]
        column = spod['columns'].get(colkey)
    
    column_by_name = None
    if column is None and name is not None and len(name) > 0:
        hits = list(filter(lambda c: c['interface-id+'] == system_key and c['name'] == name, spod['columns'].values()))
        if len(hits) == 1:
            column_by_name = hits[0]
        else:
            assert len(hits) == 0, f"Found several matching columns for name {name} in interface {key}! {hits}"
    
    if column is not None and column_by_name is not None and column_by_name != column:
        logger.error(f"Column reference and name error! {column} vs {column_by_name}")
        
    attributes = set()
    attr_ref = row[0].value
    if attr_ref is not None and ':' in attr_ref:
        index = attr_ref.rfind(':')
        attr_key = attr_ref[index+1:]
        attr = spod['attributes'].get(attr_key)
        assert attr is not None, f"Attribute {attr_key} not found"
        attributes.add(attr_key)
        logger.debug(f"Using attribute {attr_key} '{attr['techname']}' '{attr['name'].get('en')}' for row {row[0].row}. Line: {nonempty_cells_values}")
    
    if len(attributes) < 1:
        ename = row[1].value
        aname = row[2].value
        if ename is not None and aname is not None and len(ename) > 0 and len(aname) > 0:
            logger.warning(f"No attribute found. Using name lookup: {ename}.{aname}")
            candidate_attributes = filter(lambda t: t[1]['name']['de'] == aname, spod['attributes'].items())
            for akey, attr in candidate_attributes:
                ekey = attr['entity']
                entity = spod['entities'][ekey]
                if entity['name']['de'] == ename:
                    attributes.add(akey)
                    logger.debug(f"Sucessfully resolved attribute {ename}.{aname} to {akey}")
    
    if len(attributes) < 1 and len(aonline) > 1:
        """Create a new attribute for the line"""
        attr_id = new_attribute_cache.get(row[0].row)
        if attr_id is None:
            aname =  f"R{row[0].row}"
            description = f"Placeholder attribute for row {row[0].row} by system {spod['systems'][system_key]['name']}: {nonempty_cells_values}"
            logging.warning(f"No attribute mapping found for row {row[0].row} -> create unassigned attribute '{aname}'. Line: {nonempty_cells_values}")
            attr_id = 'ATTR' + str(next(new_element_id_generator))
            line_attribute = attr2js(None)
            line_attribute['techname'] = aname
            line_attribute['name'] = { k: aname for k in spod['languages'].keys() }
            line_attribute['tooltip'] = { k: '' for k in spod['languages'].keys() }
            line_attribute['descr'] = { k: description for k in spod['languages'].keys() }
            line_attribute['entity'] = default_entity
            line_attribute['domain'] = default_domain
            line_attribute['columnsmapped+'] = {} # clear default
            line_attribute['sourceref'] = {} # clear default
            line_attribute['examples'] = []
            new_attribute_cache[row[0].row] = attr_id
            result.append( ('c', attr_id, line_attribute, None) )
        attributes.add(attr_id)
    
    if column is None:
        fresh = new_column_cache.get(name)
        if fresh is not None:
            column = fresh
            
    if column is not None:
        updated = update_column(column, name, tech, row, attributes)
        if updated is not None:
            logger.debug(f"Found column reference for update: {key}")
            result.append( ('u', colkey, updated, column) )
    else:
        if name is not None or tech is not None:
            logging.debug(f"Creating new column '{name}' for system '{system_key}")
            if name is None:
                name = tech
            tables = spod['systems'][system_key]['tables+']
            if len(tables) > 0:
                logging.debug("Using first table to add new columns")
                table_id = tables[0]
            else:
                table_id = new_table_cache.get(system_key + '-new')
                if table_id is None:
                    table_id = 'TABL' + str(next(new_element_id_generator))
                    new_table = { 'name': 'main', 'interface-id': system_key,
                                 'entitiesmapped': [],
                                 'relationsmapped': [],
                                 'columnsmapped': [],
                                 'referencedby': [],
                                 'prefix': None, 'descr': create_statement, 
                                 'uc': user, 'dc': stamp,
                                 # hack
                                 'interface-id+': system_key,
                                }
                    add_defaults(new_table)
                    new_table_cache[system_key + '-new'] = table_id
                    result.append( ('c', table_id, new_table, None) )
        
        
            tkey = f'{table_id}:{name}'
            column = new_column_cache.get(tkey)
            if column is None:
                column = { 'name': name, 'table-id': table_id, 'interface_col_id': tech,
                        'uc': user, 'dc': stamp,
                        'attributesmapped': list(attributes),
                        'mandatory': False, 'datatype': 'unknown', 'format': None, 'domain': default_domain, 
                        'descr': create_statement + f" row:{row[0].row}",
                    }

                add_defaults(column)
                new_column_cache[tkey] = column
                result.append( ('c', 'COLU' + str(next(new_element_id_generator)), column, None) )
                logging.info(f"New column '{name}' mapped to attributes {attributes}")
            else:
                logging.error(f"How the heck! Value '{tkey}' is a duplicate on {row[0].row}")
                assert False
                    
    for t in result:
        type_id = t[1][:4]
        # Crucial part! Generate a unique id for each and every element
        srcname = f"EMI:{source_excel_file.stem.replace(' ','_')}:{row[0].row} {type_id}:{system_key}"
        element = t[2]
        ref = element.get('sourceref')
        if ref is None:
            element['sourceref'] = { SOURCE_KEY: [ srcname, change_stamp ] }
        else:
            ref[SOURCE_KEY] = [ srcname, change_stamp ]
        add_defaults(element)
    
    return result

Default domain is DOMA78


In [32]:
processed = []
all_changes = []

# TODO switch from system scan to column scan

for skey, system in spod['systems'].items():
    name_col_index = column_mapping.get(system['name'])
    if name_col_index is not None:
        key_col_index = column_mapping.get(skey + ':FQN')
        tech_col_index = column_mapping.get(skey + ':REF')

        data_iterator = mappings.iter_rows(min_row=first_data_row, values_only=False)
        changes = []
        for row in tqdm(data_iterator, desc=f"Processing rows of system {system['name']}", unit=" Row", dynamic_ncols=True):
            value = row[name_col_index].value
            key = None if key_col_index is None else row[key_col_index].value
            tech = None if tech_col_index is None else row[tech_col_index].value
            
            elements = process_system_row(spod, skey, key=key, name=row[name_col_index].value, tech=tech, row=row)
            changes += elements
        if len(changes) > 0:
            logging.info(f"Applying {len(changes)} element changes for system {system['name']}")
            all_changes += changes
        processed.append(f"{system['name']} [{skey}]")
    else:
        logging.warning(f"System '{system['name']}' not present")

Processing rows of system ADDOK_MATCO / COFAQ: 0 Row [00:00, ? Row/s]

DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 12. Line: ['REFANT', 'GTIN actifs', 'GTIN / EAN13', 'EAN', '*Gencod (EAN 13)', 'GENCOD']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 13. Line: ['GTIN13', 'ean', 'Gencod Produit', 'Ean', 'GTIN13', 'GENCOD', 'GENCOD', 'CODE EAN', 'GTIN Fournisseur', 'EAN']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 14. Line: ['ean_remplacement', "Gencod de l'article supprimé /remplacé (ancien gencod)"]
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 25. Line: ['POIDSP', 'poids_en_kg', 'Poids net (kg)', 'POIDS', 'Poids net', 'Poids du produit nu (en kg)*', '*Poids du produit sans emballage (en kg)', 'Poids net UVC (kg)']
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 26. Line: ['Poids net (gr)', '*Sans emballage (en kg)']
DEBUG:ExcelMappingLogger:

Processing rows of system BATI: 0 Row [00:00, ? Row/s]

DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 12. Line: ['REFANT', 'GTIN actifs', 'GTIN / EAN13', 'EAN', '*Gencod (EAN 13)', 'GENCOD']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 13. Line: ['GTIN13', 'ean', 'Gencod Produit', 'Ean', 'GTIN13', 'GENCOD', 'GENCOD', 'CODE EAN', 'GTIN Fournisseur', 'EAN']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 14. Line: ['ean_remplacement', "Gencod de l'article supprimé /remplacé (ancien gencod)"]
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 25. Line: ['POIDSP', 'poids_en_kg', 'Poids net (kg)', 'POIDS', 'Poids net', 'Poids du produit nu (en kg)*', '*Poids du produit sans emballage (en kg)', 'Poids net UVC (kg)']
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 26. Line: ['Poids net (gr)', '*Sans emballage (en kg)']
DEBUG:ExcelMappingLogger:

Processing rows of system Bricoman: 0 Row [00:00, ? Row/s]

...

DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 12. Line: ['REFANT', 'GTIN actifs', 'GTIN / EAN13', 'EAN', '*Gencod (EAN 13)', 'GENCOD']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 13. Line: ['GTIN13', 'ean', 'Gencod Produit', 'Ean', 'GTIN13', 'GENCOD', 'GENCOD', 'CODE EAN', 'GTIN Fournisseur', 'EAN']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 14. Line: ['ean_remplacement', "Gencod de l'article supprimé /remplacé (ancien gencod)"]
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 25. Line: ['POIDSP', 'poids_en_kg', 'Poids net (kg)', 'POIDS', 'Poids net', 'Poids du produit nu (en kg)*', '*Poids du produit sans emballage (en kg)', 'Poids net UVC (kg)']
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 26. Line: ['Poids net (gr)', '*Sans emballage (en kg)']
DEBUG:ExcelMappingLogger:

Processing rows of system POINTE P_SIKA_FRANCE: 0 Row [00:00, ? Row/s]

DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 12. Line: ['REFANT', 'GTIN actifs', 'GTIN / EAN13', 'EAN', '*Gencod (EAN 13)', 'GENCOD']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 13. Line: ['GTIN13', 'ean', 'Gencod Produit', 'Ean', 'GTIN13', 'GENCOD', 'GENCOD', 'CODE EAN', 'GTIN Fournisseur', 'EAN']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 14. Line: ['ean_remplacement', "Gencod de l'article supprimé /remplacé (ancien gencod)"]
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 25. Line: ['POIDSP', 'poids_en_kg', 'Poids net (kg)', 'POIDS', 'Poids net', 'Poids du produit nu (en kg)*', '*Poids du produit sans emballage (en kg)', 'Poids net UVC (kg)']
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 26. Line: ['Poids net (gr)', '*Sans emballage (en kg)']
DEBUG:ExcelMappingLogger:

Processing rows of system PROLIANS_348-SIKA FRANCE: 0 Row [00:00, ? Row/s]

DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 12. Line: ['REFANT', 'GTIN actifs', 'GTIN / EAN13', 'EAN', '*Gencod (EAN 13)', 'GENCOD']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 13. Line: ['GTIN13', 'ean', 'Gencod Produit', 'Ean', 'GTIN13', 'GENCOD', 'GENCOD', 'CODE EAN', 'GTIN Fournisseur', 'EAN']
DEBUG:ExcelMappingLogger:Using attribute ATTR308 'GTIN__EAN/UPC_' 'GTIN (EAN/UPC)' for row 14. Line: ['ean_remplacement', "Gencod de l'article supprimé /remplacé (ancien gencod)"]
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 25. Line: ['POIDSP', 'poids_en_kg', 'Poids net (kg)', 'POIDS', 'Poids net', 'Poids du produit nu (en kg)*', '*Poids du produit sans emballage (en kg)', 'Poids net UVC (kg)']
DEBUG:ExcelMappingLogger:Using attribute ATTR1304 'NETTOGEWICHT' 'Net weight' for row 26. Line: ['Poids net (gr)', '*Sans emballage (en kg)']
DEBUG:ExcelMappingLogger:

In [33]:
len(all_changes), len(processed), processed

(1502,
 14,
 ['ADDOK_MATCO / COFAQ [INTF4594]',
  'BATI [INTF5420]',
  'Bricoman [INTF5421]',
  'CMEM_SIKA [INTF4595]',
  'CTX-LMFR-FR [INTF5422]',
  'Champs Alkemics [INTF5423]',
  'Champs SMARTREF [INTF5424]',
  'Chausson_Tarifs SIKA [INTF2187]',
  'GEDIMAT_Matrice_ARTICLE-SIKA [INTF2193]',
  'GEDIMAT_TARIF [INTF2194]',
  'LA PDB_Création [INTF2196]',
  'MR BRICOLAGE [INTF5425]',
  'POINTE P_SIKA_FRANCE [INTF2199]',
  'PROLIANS_348-SIKA FRANCE [INTF2200]'])

## Add new systems

In [34]:
def add_system(workbook, column: str, name: str) -> dict:
    new_system = {
        'name': name,
    }
    system_id = 'INTF' + str(next(new_element_id_generator))
    return ('c', system_id, new_system, None)

### Scan all columns for a 'New' comment

# Apply changes to spod

In [35]:
changes = []
creations = 0
updates = 0

spod_updated = copy.deepcopy(spod)
with tqdm(total=len(all_changes), unit=' Operation') as progress:
    for action, key, element, previous in all_changes:
        progress.set_description(f"Processing {action} on {key}")
        change = { 
            'action': action, 
            'key': key, 
        }
        message = f"({action} on {key}"
        if action == 'c':
            change['new'] = element
            if key.startswith('COLU'):
                message = f"Creating column {key}"
                spod_updated['columns'][key] = element
                creations += 1
            if key.startswith('TABL'):
                message = f"Creating table {key} for system '{element['name']}' {element['interface-id']}"
                spod_updated['tables'][key] = element
                creations += 1
            if key.startswith('ATTR'):
                message = f"Creating attribute {key} for entity '{element['entity']}'"
                spod_updated['attributes'][key] = element
                creations += 1
                
        if action == 'u':
            change['new'] = element
            change['current'] = previous
            if key.startswith('COLU'):
                message = f"Updating column {key}"
                spod_updated['columns'][key] = element
                updates += 1
                
        logging.debug(message)
        change['message'] = message
        changes.append(change)
        progress.update(1)
        time.sleep(0.003)

  0%|          | 0/1502 [00:00<?, ? Operation/s]

In [36]:
creations, updates

(1502, 0)

In [37]:
changelog_file = Path('changelog.json') 
with open(changelog_file, 'w') as out:
    json.dump(changes, out)
logger.info(f"Wrote {len(changes)} changes to {changelog_file}")

INFO - Wrote 1502 changes to changelog.json
INFO:ExcelMappingLogger:Wrote 1502 changes to changelog.json


### Upgrade spod structure

In [38]:
# Patch JSON
#spod_updated['actorroles'] = {}

In [39]:
%%script false --no-raise-error

for element in spod_updated['entities'].values():
    ex = element['examples']
    if isinstance(ex, dict):
        print(f"Transforming example {ex} to array")
        element['examples'] = [ ]

In [40]:
# fix SPOD
spod_updated['entities'].pop('ENTI116', None)
spod_updated['entities'].pop('ENTI364', None)
#ATTR397
#ATTR4700
#ATTR1309

{'name': {'de': 'Kit', 'en': 'Kit', 'fr': 'Kit'},
 'shortname': None,
 'descr': {'de': 'Mehrere Einheiten von mehreren Artikeln zusammen verpackt und angeboten.\n\nMuss mindestens 2 verschiedene Artikel referenzieren.\nMuss immer eine eigene Verpackung haben\n\nBeispiel:',
  'en': 'Several units of several items packed and offered together.\n\nMust reference at least 2 different items.\nMust always have its own packaging',
  'fr': 'Plusieurs unités de plusieurs articles emballées et offertes ensemble.\n\nDoit référencer au moins 2 articles différents.\nDoit toujours avoir son propre emballage'},
 'tooltip': {'de': '', 'en': '', 'fr': ''},
 'category': 'CATG7',
 'exptuple#': None,
 'prefix': None,
 'supertypeentity': 'ENTI103',
 'subtypellevel+': 3,
 'uc': 'stb',
 'dc': '2021-11-10 12:37:32 UTC',
 'um': 'sys',
 'dm': '2022-03-21 11:29:08.165298',
 'minzoomlevel': None,
 'maxzoomlevel': None,
 'publstatus': None,
 'icon': {'type': None, 'reference': None},
 'synonyms': [{'de': 'Diverse A

## Store changed SPOD 

In [41]:
updated = Path(MODEL).with_suffix('.import.json')
with open(updated, 'w') as out:
    json.dump(spod_updated, out, indent=4)
print(f"Wrote {updated}")

Wrote /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.json


# Try to merge

In [42]:
from LOAD_MODELS.LOAD_INFRA import mergedbs
from SSOT_db.SQL_INFRA import dbConnect
from SSOT_db.IM_JSON.jsbase import JSModel

In [43]:
sourcedb = Path(MODEL).with_suffix('.db')
assert sourcedb.is_file(), f"Source database '{sourcedb} is missing'"

tempdb = sourcedb.with_suffix('.import.db')
# clean slate
tempdb.unlink(missing_ok=True)

shutil.copy(sourcedb, tempdb)
assert tempdb.is_file()

In [44]:
mappingfile = Path(MAPPING)

In [45]:
from SSOT_infra import parameters
print(f"Merging updated json into existing DB")
spod_updated['_imprint_']['git-revision'] = parameters.read_git_description(sourcedb.parent)
new_model = JSModel(spod_updated)
#mergedbs.checkjsonmodel(pmodel=new_model, pverbose=True)

Merging updated json into existing DB


In [46]:
from contextlib import closing

with closing(dbConnect.openDB(tempdb)):
    logger.info(f"Merging JSON into database {tempdb} for source {SOURCE_KEY}")
    mergeresult = mergedbs.mergejson2sql(new_model, psrcname=SOURCE_KEY, pverbose=True, pcheckonly=False)

mergeresult

INFO - Merging JSON into database /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.db for source Excel-Mapping-Import-Notebook
INFO:ExcelMappingLogger:Merging JSON into database /Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.db for source Excel-Mapping-Import-Notebook


Processing LANG:   0%|                                                                                        …

Processing PHYU:   0%|                                                                                        …

Processing DATY:   0%|                                                                                        …

Processing STFO:   0%|                                                                                        …

...

Processing COLU:   0%|                                                                                        …

Processing BURU:   0%|                                                                                        …

Processing DIAG:   0%|                                                                                        …

ERROR:root:*** insert-error: ID = "ATTR4699" 
Cannot insert into attributes tuple (565, 200, 110, 'HANDEL_STARTZEITPUNKT', 'Handel-Startzeitpunkt', 6, '', 'Datumsangabe für Gültigkeit von Informationen für den Datenaustausch zwischen Hersteller und Handel und für die Gültigkeit von Artikeln.\n\nZu beachten:\nMit der Datumsangabe (gültig ab ...) kann der Hersteller frühzeitig über künftige Neuanlagen, Änderungen und Korrekturen von Artikeln informieren. Das Datum gilt nur auf Artikelebene und nicht für die ganze Nachricht und auch nicht für den Preis.\nDas Gültig ab Datum ist im Zusammenhang mit dem Bewegungskennzeichen NEU unveränderlich wie ein Geburtsdatum und ändert sich nicht mit der Herausgabe einer neuen Preisliste.', 'FALSE', 'FALSE', 'FALSE', 'FALSE', 'FALSE', 'FALSE', 'stb', '2022-03-02 14:53:55 UTC', None, None).
UNIQUE constraint failed: attributes.attr_displ_name, attributes.attr_enti_id from insert into attributes (attr_id=565, attr_enti_id=200, attr_doma_id=110, attr_tech

AssertionError: 

## Load restore json from updated DB

In [ ]:
from SSOT_db.IM_JSON.jsmodel import sql2json
from SSOT_infra.parameters import read_git_description

with closing(dbConnect.openDB(tempdb)):
    loadedjson = JSModel(pmodel=sql2json(pdbname=str(tempdb)))
loadedjson.jsmodel['_imprint_']['git-revision'] = read_git_description(sourcedb.parent)

In [ ]:
regen_file = tempdb.with_suffix('.regen.json')
with open(regen_file, 'w') as out:
    json.dump(loadedjson.jsmodel, out, indent=4)
logger.info(f"Wrote regenerated JSON to {regen_file}")

In [ ]:
spod_re = loadedjson.jsmodel
for category in spod_re.keys():
    if category in ['entities', 'attributes', 'systems', 'tables', 'columns']:
        print(f"Count difference for '{category}':  {len(spod[category])} -> {len(spod_re[category])}")
        for key, element in spod_re[category].items():
            pass